# Lab 2: Living in the Wumpus World: Logical Agents with propositional logic

In this lab we will learn how to use propositional logic and knowledge bases, which you created in Lab 1 to create a simple agent to traverse the Wumpus World.

The lab includes:

- Implementing an agent with a knowledge base of the Wumpus World, can query that knowledge base for which tiles are safe to visit.

The content of this lab relates strongly to the material of Chapter 6 (Logical Agents) in the course book (Artificial Intelligence: A Modern Approach).
The enviornment we will tackle in this lab is called the Wumpus World and is covered in the course book as well (Chapter 6.2)

# Implementation of the Wumpus World

Below you will find a code implementation of the Wumpus World along with a few helpful definitions and functions.
The Wumpus World is fully implemented since this lab is about building a propositional logic knowledge base system to build an AI around.
However, since the AI will interact with the Wumpus World, being familiar with the code and functions will help you use them later.

In [ ]:
#Imports
import random
import itertools

# Directions
RIGHT = 0
DOWN  = 1
LEFT  = 2
UP    = 3

# How much the agent will move if it takes a step forward for each direction
direction_step = [(1,0), (0,-1), (-1,0), (0,1)]

# Objects
AGENT   = "a"
WUMPUS  = "w"
PIT     = "p"
GOLD    = "g"

# Actions
NONE = ""
FORWARD = "forward"
TURN_LEFT = "turn_left"
TURN_RIGHT = "turn_right"
GRAB = "grab"
SHOOT = "shoot"
CLIMB = "climb"

class WumpusWorld:
    
    def __init__(self):
        self.game_over = False
        self.agent_dir = RIGHT
        self.agent_has_arrow = True
        self.agent_has_gold = False
        self.wumpus_screams = False
        self.non_start_squares = list(itertools.product([0,1,2,3],[0,1,2,3]))
        self.non_start_squares.pop(0)
        self.cave = [[[],[],[],[]],[[],[],[],[]],[[],[],[],[]],[[],[],[],[]]]
        self.cave[0][0].append(AGENT)
        self.agent_pos = (0,0)
        self.random_place(WUMPUS)
        self.random_place(GOLD)
        for square in self.non_start_squares:
            if random.random() < 0.2:
                self.cave[square[0]][square[1]].append(PIT)
        self.performance = 0
        self.last_action = NONE
    
    def random_place(self,object):
        '''
        Randomly places an object into a tile other than the start.
        Args:
            object (any): The object (typically WUMPUS or GOLD) to place
        '''
        pos = random.choice(self.non_start_squares)
        self.cave[pos[0]][pos[1]].append(object)
        
    def show(self):
        '''
        Prints the Wumpus World cave and current agent performance.
        '''
        for y in range(3,-1,-1):
            for x in range(0,4):
                [print(o,end="") for o in self.cave[x][y]]
                print("\t",end="")
            print()
        print(f"Performance: {self.performance}")

    def actuate(self, action):
        '''
        If the game is still running, the input action is performed and the game state is moved forward.
        Args:
            action (str): The action the agent should perform
        '''
        # If the game is over, nothing can be done
        if self.game_over:
            return
        
        # store the action that was performed, decrease the performance, and remove scream from memory
        self.last_action = action
        self.performance = self.performance-1
        self.wumpus_screams = False
        
        # If the NONE action is performed, do nothing
        if action == NONE:
            return
        
        # FORWARD action: Move the agent one tile in its current direction.
        # End the game and lower performance if it steps into a PIT or WUMPUS
        if action == FORWARD:
            x, y = self.agent_pos
            self.cave[x][y].remove(AGENT)
            xp, yp = direction_step[self.agent_dir]
            x = min(max(x+xp,0),3)
            y = min(max(y+yp,0),3)
            self.agent_pos = (x,y)
            if WUMPUS in self.cave[x][y] or PIT in self.cave[x][y]:
                self.game_over = True
                self.performance = self.performance-1000
            else:
                self.cave[x][y].append(AGENT)
            return
        
        # TURN_LEFT action: Change the agents direction 90 degrees counter-clockwise
        if action == TURN_LEFT:
            self.agent_dir = (self.agent_dir-1)%4
            return
        # TURN_RIGHT action: Change the agents direction 90 degrees clockwise
        if action == TURN_RIGHT:
            self.agent_dir = (self.agent_dir+1)%4
            return
        # GRAB action: If the agent is in the same tile as the gold, remove it from the cave and add it to the agent
        if action == GRAB:
            x, y = self.agent_pos
            if GOLD in self.cave[x][y]:
                self.agent_has_gold = True
                self.cave[x][y].remove(GOLD)
            return
        # SHOOT action: If the agent has the arrow,
        #    remove it, decrease performance, and, 
        #    if there is a WUMPUS infront of the agent, remove it and add scream perception 
        if action == SHOOT:
            if not self.agent_has_arrow:
                return
            self.agent_has_arrow = False
            self.performance = self.performance-10
            x, y = self.agent_pos
            if self.agent_dir == RIGHT:
                hit_squares = [(min(x+1,3),y),(min(x+2,3),y),(min(x+3,3),y)]
            elif self.agent_dir == DOWN:
                hit_squares = [(x,max(y-1,0)),(x,max(y-2,0)),(x,max(y-3,0))]
            elif self.agent_dir == LEFT:
                hit_squares = [(max(x-1,0),y),(max(x-2,0),y),(max(x-3,0),y)]
            elif self.agent_dir == UP:
                hit_squares = [(x,min(y+1,3)),(x,min(y+2,3)),(x,min(y+3,3))]
            for x, y in hit_squares:
                if WUMPUS in self.cave[x][y]:
                    self.cave[x][y].remove(WUMPUS)
                    self.wumpus_screams = True
            return
        
        # CLIMB action: If the agent is at the starting tile, end the game and add performance if the agent has the gold
        if action == CLIMB:
            if self.agent_pos == (0,0):
                if self.agent_has_gold:
                    self.performance = self.performance+1000
                self.cave[0][0].remove(AGENT)
                self.agent_pos = None
                self.agent_dir = None
                self.game_over = True
            return
    
    def perceive(self):
        '''
        Returns for each percept whether that percept was sensed after the last action.
        '''
        x,y = self.agent_pos
        stench = False
        breeze = False
        for sx,sy in adjacent_tiles(x,y):
            if WUMPUS in self.cave[sx][sy]:
                stench = True
            if PIT in self.cave[sx][sy]:
                breeze = True
        glitter = GOLD in self.cave[x][y]
        bump = False
        if self.last_action == FORWARD:
            if self.agent_dir == RIGHT and x == 3:
                bump = True
            elif self.agent_dir == DOWN and y == 0:
                bump = True
            elif self.agent_dir == LEFT and x == 0:
                bump = True
            elif self.agent_dir == UP and y == 3:
                bump = False
        scream = self.wumpus_screams
        return(stench, breeze, glitter, bump, scream)

def adjacent_tiles(x, y):
    '''
    Helper function that, given a position, returns all tiles that are adjacent to it.
    Args:
        x (int): The x coordinate of the position
        y (int): The y coordinate of the position
    '''
    tiles = []
    for sx,sy in [(x-1,y),(x+1,y),(x,y-1),(x,y+1)]:
        if sx<4 and sx>-1 and sy<4 and sy>-1:
            tiles.append((sx,sy))
    return tiles

# INSERT THE IMPLEMENTED AND TESTED CONTENT OF LAB 1 HERE


# Using logic to understand the Wumpus World

Now that we have a working Propositional Logic, Knowledge Base, and Query system in place, we can finally start using it to create an agent.
Knowledge-based agents can come in many forms, but the most typical behavior is to dump all the relevant knowledge of the world into a knowledge base and then query the knowledge base for what action to perform next.
The percepts from the environment is then used to add new knowledge into the knowledge base to help with the next query.

Doing all of this can require very extensive knowledge bases and complex agents, which is beyond the scope of this assigment.
Instead we will use a very simple procedure for our agent:
- If the agent has the gold we know there must be safe path back to the entrance, take actions to follow that path back and then use CLIMB to finish the game.
- If there is glitter on the current tile, use the GRAB action to pick it up.
- Otherwise, query every tile for whether its safe or not.
    - If there is a safe tile that has not yet been visited, calculate a path to it and then follow that path.
        - If there is no path, calculate a path to the next unvisited safe tile.
    - If all safe tiles have been visited or are unreachable, find a path back to the start tile and then use CLIMB to finish the game.

As you can see this agent is very simple, not even considering using the SHOOT action, and assumes that all tiles that it doesn't know for sure to be unsafe.
For now though, this is enough.
Before we can start creating an agent, we need a knowledge base for it to query.
There are many things we know about the Wumpus World, many of which we could turn into logic and put into the knowledge base.
However, the more things we put in the knowledge base the longer the query will take, so we want to stick with only the essentials.
For example, we know that there can only be one WUMPUS and that each tile (except the start) has a 20% chance of being a PIT.
But the 20% chance is difficult to encode in propositional logic and not really relevant to our query, since we only care about things we know for sure.

So what do we put in the knowledge base?
We want to look for safe squares, so wee need to put a sentence for each tile defining what it means for that tile to be safe (there is no WUMPUS and no PIT there).
We also need some way to figure out whether there is a WUMPUS on a tile, so we need to add that having a WUMPUS on a tile means that the surrounding tiles all have stench.
The same goes for PIT and breeze.
Of course we also know some things about the starting tile.

Then we need a way to add new knowledge into the knowledge base based on the percepts that are encountered.

So now we need to create two functions.
One that returns the knowledge base about all relevant things we know at the start of each game, and one that given the percepts, the agent's position when those percepts were recieved, and a knowledge base, adds new relevant knowledge to the knowledge base.
A skeleton for each function is provided below

*Hint*: Remember that each sentence needs to be created for each tile and that each tile needs its own versions of all symbols. Try to think of a systematic way to name the symbols based on their tile coordinates.

In [ ]:
def initialWumpusKB():
    '''
    Returns a knowledge base of sentences known to be true in all initial Wumups Worlds
    Returns ({any}): A set of logic sentences
    '''
    # Write a function that creates a set of all relevant starting knowledge.
    return {}

    
def addWumpusKnowledge(percepts, agent_pos, KB):
    '''
    Updates the given knowledge base by adding (and potentially removing) sentences from the knowledge base,
    based on the Wumpus World agent's position and perceptions.
    Args:
        percepts ([bool]): A list of whether each percept is active after the last action
        agent_pos (int, int): The tile currently occupied by the agent
        KB ({any}): The knowledge base of the current Wumpus World to be updated
    '''
    stench, breeze, glitter, bump, scream = percepts
    # Write a function that given percepts adds the relevant knowledge to the KB

## Creating the agent

With knowledge base in hand it's time to create our simple naive agent.

Reading the description of the agent ahead, you probably noticed that it requires a search algorithm to find a path from one point to another.
Since search algorithms aren't covered in this course a simple, implementation of the shortest path algorithm **A\*** has been implemented in the next code block for you to use.
It takes the start tile, goal tile, and a list of all safe tiles and then returns the next tile to walk to.
Even though search-algorithms like **A\*** are not covered in this course, you can read more about them in Chapter 3 (Solving Problems by Searching) of the course book (Artificial Intelligence: A Modern Approach).

You also recieve a function to run the game given a class to return actions (an agent), and a skeleton for the agent you will implement based on the earlier description.

In [ ]:
# An implementation of a priority queue that is necessary for calculating A*
import heapq

def a_star_on_tiles(start, goal, safe_tiles):
    '''
    Takes a start tile, end tile, and a list of safe tiles and returns the next step along the shortest path to the goal.
    Assumes that the world is a grid where each tile is connected to the adjacent tiles (not diagonally)
    Args:
        start (int, int): The coordinates of the start tile
        goal (int, int): The tile to find a shortest path towards
        safe_tiles [(int, int)]: A list of the tiles that can be traversed (should include start and goal)
    Returns ((int, int)/None): The next tile in the shortest path if one exists
    '''
    frontier = []
    heapq.heappush(frontier, (0, start))
    came_from = {}
    cost_so_far = {}
    came_from[start] = None
    cost_so_far[start] = 0
    
    # Run A* to get a dictionary containing for each tile, the previous tile along the path
    while len(frontier) > 0:
        distance, (x, y) = heapq.heappop(frontier)
        
        if (x,y) == goal:
            break
        
        for neighbor in [n for n in adjacent_tiles(x,y) if n in safe_tiles]:
            new_cost = cost_so_far[(x,y)] + 1 #Can be adapted to handle turning time
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                priority = new_cost + abs(x-neighbor[0]) + abs(y-neighbor[1])
                heapq.heappush(frontier, (priority, neighbor))
                came_from[neighbor] = (x,y)
    
    # Walk backward along the path until the first tile beyond the start is found
    # If the start tile is unreachable, return None
    current = goal
    while True:
        if current not in came_from:
            return None
        if came_from[current] == start:
            break
        current = came_from[current]
    
    return current

In [ ]:
def play_game(agent, show=True):
    '''
    Initiates and runs a random Wumpus World by calling the Agent.act() function to get actions
    Args:
        agent (any): An object which has implemented an act function
        show (bool): Whether to print the states of the game or not
    '''
    w = WumpusWorld()
    while not w.game_over:
        if show:
            w.show()
        w.actuate(agent.act(w.perceive(), w.agent_pos, w.agent_dir))
    if show:
        w.show()

In [ ]:
class SimpleAgent():
    '''
    Creates a simple knowledge-based agent for acting in the Wumpus World
    '''
    
    def __init__(self):
        pass
        # Initialize the agents internal parameters including the knowledge base
    
    def act(self, percepts, position, direction):
        '''
        Returns an action based on the agents percepts and position in this time step
            Args:
                percepts ([bool]): A list of, for each percept, whether it is active in this time step
                position (int, int): The coordinates of the tile the agent is currently in
                direction (int):  An int representing the direction the agent is currently facing
            Returns (int): An int representing the action to be taken (see the first cell for definitions)
        '''
        #Implement the earlier described procedure for a simple naive agent
        raise NotImplementedError('The SimpleAgent has not been implemented yet')

In [ ]:
# Execute this cell to play one game over the Wumpus World with the SimpleAgent

play_game(SimpleAgent())